In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.linear_model import Lars
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load enzyme data
csv_path = Path("Wold_DDGTS.csv")
if not csv_path.exists():
    csv_path = Path(r"C:\Users\Mikey\Desktop\CHE891_Classwork\Process_Design\Class_Jupyter_Files\Wold_DDGTS.csv")

df = pd.read_csv(csv_path)
print("Data shape:", df.shape)
print(df.head())

# Predict DDGTS from physicochemical descriptors
feature_cols = ["PIE", "PIF", "DGR", "SAC", "MR", "Lam", "Vol"]
target_col = "DDGTS"

X = df[feature_cols].values
y = df[target_col].values

# Small dataset: tune sparsity using Leave-One-Out CV
loo = LeaveOneOut()
max_vars = X.shape[1]
cv_r2 = []

for n_vars in range(1, max_vars + 1):
    model_cv = Pipeline([
        ("scaler", StandardScaler()),
        ("lars", Lars(n_nonzero_coefs=n_vars))
    ])
    y_hat_loo = cross_val_predict(model_cv, X, y, cv=loo)
    cv_r2.append(r2_score(y, y_hat_loo))

best_n_vars = int(np.argmax(cv_r2)) + 1
print("\nLOOCV R^2 by # active variables:")
for n_vars, score in enumerate(cv_r2, start=1):
    print(f"  {n_vars:>2d} vars: {score:+.4f}")
print(f"\nBest sparsity: {best_n_vars} active variables")

# Fit final LARS model on full data with selected sparsity
model = Pipeline([
    ("scaler", StandardScaler()),
    ("lars", Lars(n_nonzero_coefs=best_n_vars))
])
model.fit(X, y)
y_pred = model.predict(X)

r2_train = r2_score(y, y_pred)
rmse_train = mean_squared_error(y, y_pred) ** 0.5
print(f"\nFinal model training R^2: {r2_train:.4f}")
print(f"Final model training RMSE: {rmse_train:.4f}")

# Report selected coefficients in original feature order
coef = model.named_steps["lars"].coef_
coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": coef,
    "selected": np.abs(coef) > 1e-12
})
print("\nLARS coefficients:")
print(coef_table)


Data shape: (19, 8)
    PIE   PIF   DGR    SAC     MR   Lam    Vol  DDGTS
0  0.23  0.31 -0.55  254.2  2.126 -0.02   82.2    8.5
1 -0.48 -0.60  0.51  303.6  2.994 -1.24  112.3    8.2
2 -0.61 -0.77  1.20  287.9  2.994 -1.08  103.7    8.5
3  0.45  1.54 -1.40  282.9  2.933 -0.11   99.1   11.0
4 -0.11 -0.22  0.29  335.0  3.458 -1.19  127.5    6.3

LOOCV R^2 by # active variables:
   1 vars: +0.3937
   2 vars: +0.2577
   3 vars: +0.2482
   4 vars: +0.2318
   5 vars: -0.0548
   6 vars: -0.4921
   7 vars: -0.5309

Best sparsity: 1 active variables

Final model training R^2: 0.5016
Final model training RMSE: 1.8950

LARS coefficients:
  feature  coefficient  selected
0     PIE     0.000000     False
1     PIF     1.729016      True
2     DGR     0.000000     False
3     SAC     0.000000     False
4      MR     0.000000     False
5     Lam     0.000000     False
6     Vol     0.000000     False


LARS would eliminate any coefficients equal to zero, but in this notebook I am predicting DDGTS from a set on information on each amino acid individualy, so I don't understand why eliminating amino acids would improve the fit. Lars is selecting physiochemical descriptors as the variables, not amino acids.